### Setup

In [43]:
# Jupyter autoreload
%load_ext autoreload
%autoreload 2

# standard library
import os
import sys

# third-party
import numpy as np
import pandas as pd

# local path
sys.path.append(os.path.abspath("../src"))

# local imports
import pis.returns as returns

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Create Sample Price Data

In [44]:


data = {
    "SP500": [100, 101, 102, 101, 103],
    "BONDS": [100, 100.5, 101, 101.2, 101.5],
    "GOLD": [100, 99, 100, 101, 102]
}

df = pd.DataFrame(data)
df

,SP500,BONDS,GOLD
0,100,100.0,100
1,101,100.5,99
2,102,101.0,100
3,101,101.2,101
4,103,101.5,102


### Compute Simple Returns

In [45]:
simple_returns = returns.compute_simple_returns(df)
simple_returns

,SP500,BONDS,GOLD
0,NaN,NaN,NaN
1,0.010000,0.005000,-0.010000
2,0.009901,0.004975,0.010101
3,-0.009804,0.001980,0.010000
4,0.019802,0.002964,0.009901


### Validate Simple Returns

In [46]:
expected = (101 / 100) - 1
actual = simple_returns.loc[1, "SP500"]

expected, actual, np.isclose(expected, actual)

(0.010000000000000009, np.float64(0.010000000000000009), np.True_)

### Compute Log Returns

In [47]:
log_returns = returns.compute_log_returns(df)
log_returns

,SP500,BONDS,GOLD
0,NaN,NaN,NaN
1,0.009950,0.004988,-0.010050
2,0.009852,0.004963,0.010050
3,-0.009852,0.001978,0.009950
4,0.019608,0.002960,0.009852


### Validate Log Returns

In [48]:
expected = np.log(101 / 100)
actual = log_returns.loc[1, "SP500"]

expected, actual, np.isclose(expected, actual)


(np.float64(0.009950330853168092), np.float64(0.009950330853168092), np.True_)

### Compare Simple vs Log Returns

In [49]:
simple_returns - log_returns

,SP500,BONDS,GOLD
0,NaN,NaN,NaN
1,0.000050,0.000012,0.000050
2,0.000049,0.000012,0.000051
3,0.000048,0.000002,0.000050
4,0.000194,0.000004,0.000049


### Compare Mean Returns

In [50]:
mean_comparison = pd.DataFrame({
    "simple_mean": simple_returns.mean(),
    "log_mean": log_returns.mean()
})
mean_comparison

,simple_mean,log_mean
SP500,0.007475,0.007390
BONDS,0.003730,0.003722
GOLD,0.005001,0.004951


### Cumulative Wealth From Simple Returns

In [51]:
wealth_simple = returns.compute_cumulative_wealth(simple_returns)
wealth_simple

,SP500,BONDS,GOLD
0,NaN,NaN,NaN
1,1.01,1.005,0.99
2,1.02,1.010,1.00
3,1.01,1.012,1.01
4,1.03,1.015,1.02


### Cumulative Wealth From Simple Returns Validation

In [52]:
expected = 101 / 100
actual = wealth_simple.loc[1, "SP500"]
expected, actual, np.isclose(expected, actual)

(1.01, np.float64(1.01), np.True_)

### Cumulative Wealth From Log Returns

In [53]:
wealth_log = returns.compute_cumulative_wealth_from_log(log_returns)
wealth_log

,SP500,BONDS,GOLD
0,NaN,NaN,NaN
1,1.01,1.005,0.99
2,1.02,1.010,1.00
3,1.01,1.012,1.01
4,1.03,1.015,1.02


In [54]:
expected = 101 / 100
actual = wealth_log.loc[1, "SP500"]
expected, actual, np.isclose(expected, actual)

(1.01, np.float64(1.01), np.True_)

### Wealth From Simple Returns vs Wealth From Log Returns

In [55]:
np.allclose(wealth_simple, wealth_log, equal_nan=True)

True

### Day 3 — Key Insights

- Returns represent local changes in value, while wealth represents accumulated capital over time.
- Capital growth is inherently multiplicative.

- Simple returns operate in a multiplicative framework: `(1 + r).cumprod()`.
- Log returns transform multiplicative growth into an additive form: `exp(cumsum(r))`.

- Both approaches reconstruct the same wealth evolution.
- Log returns are additive, which simplifies modeling and statistical analysis.

### Conceptual Understanding

Wealth represents the evolution of capital relative to the initial investment.

The underlying process is multiplicative: capital grows through the compounding of returns.

Simple returns model this directly, while log returns provide an additive representation of the same process, making them more convenient for mathematical and statistical modeling.

### Stress Testing Data

In [56]:
prices = pd.DataFrame({
    "A": [100, 50, 100],   
})

prices

,A
0,100
1,50
2,100


### Simple Returns From Stress Testing Data

In [57]:
stress_simple = returns.compute_simple_returns(prices)
stress_simple

,A
0,NaN
1,-0.5
2,1.0


### Log Returns From Stress Testing Data

In [58]:
stress_log = returns.compute_log_returns(prices)
stress_log

,A
0,NaN
1,-0.693147
2,0.693147


### Comparing Simple Returns and Log Returns From Same Stress Data

In [59]:
np.allclose(stress_simple, stress_log, equal_nan=True)

False

### Compute Wealth From Stress Simple Returns

In [60]:
stress_wealth_simple = returns.compute_cumulative_wealth(stress_simple)
stress_wealth_simple

,A
0,NaN
1,0.5
2,1.0


### Compute Cumulative Wealth From Stress Log Returns

In [61]:
stress_wealth_log = returns.compute_cumulative_wealth_from_log(stress_log)
stress_wealth_log

,A
0,NaN
1,0.5
2,1.0


### Comparing Cumulative Wealth from Stress Simple Returns And Stress Log Returns

In [62]:
np.allclose(stress_wealth_log, stress_wealth_simple, equal_nan=True)

True

### Stress Test Insight

Simple returns and log returns can differ significantly, especially under large price changes.

However, when transformed into cumulative wealth, both representations lead to the same final result.

This highlights that returns are different representations of the same underlying process, while wealth reflects the true evolution of capital.